In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Paths: replace these with the actual JRA and SPEEDY files/patterns.
# open_mfdataset accepts wildcards.
# ------------------------------------------------------------------
JRA_PATH = Path("/leonardo_scratch/fast/ICT26_ESP/ntilinin/INPUT/OMIP/tas_input4MIPs_atmosphericState_OMIP_MRI-JRA55-do-1-5-0_gr_199001010000-199012312100.nc")
SPEEDY_PATH = Path("/leonardo_scratch/fast/ICT26_ESP/ntilinin/SPEEDY_access/forcing/tas_SPEEDY_1990.nc")

JRA_VARIABLE = "tas"
SPEEDY_VARIABLE = "tas"

# Scientific comparison target:
# "jra"    -> interpolate SPEEDY to the JRA grid
# "speedy" -> interpolate JRA to the SPEEDY grid
REGRID_TO = "jra"

# Use only for a quick test before running over all files.
MAX_TIME_RECORDS = None

In [3]:
# =========================
# 1. Open datasets

def open_any(path):
    path = str(path)
    if any(char in path for char in "*?[]"):
        return xr.open_mfdataset(
            path,
            combine="by_coords",
            parallel=False,
            chunks={"time": 120},
            decode_times=True,
            use_cftime=True,
        )

    return xr.open_dataset(
        path,
        chunks={"time": 120},
        decode_times=True,
        use_cftime=True,
    )

jra = open_any(JRA_PATH)
speedy = open_any(SPEEDY_PATH)

if MAX_TIME_RECORDS is not None:
    jra = jra.isel(time=slice(0, MAX_TIME_RECORDS))
    speedy = speedy.isel(time=slice(0, MAX_TIME_RECORDS))

print("JRA")
print(jra)
print("\nSPEEDY")
print(speedy)


/scratch_local/ipykernel_2460209/2629220063.py:16: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(


JRA
<xarray.Dataset> Size: 2GB
Dimensions:    (time: 2920, bnds: 2, lat: 320, lon: 640)
Coordinates:
  * time       (time) object 23kB 1989-01-01 00:00:00 ... 1989-12-31 21:00:00
  * lat        (lat) float64 3kB -89.57 -89.01 -88.45 ... 88.45 89.01 89.57
  * lon        (lon) float64 5kB 0.0 0.5625 1.125 1.688 ... 358.3 358.9 359.4
    height     float64 8B ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds  (time, bnds) object 47kB dask.array<chunksize=(120, 2), meta=np.ndarray>
    lat_bnds   (lat, bnds) float64 5kB dask.array<chunksize=(320, 2), meta=np.ndarray>
    lon_bnds   (lon, bnds) float64 10kB dask.array<chunksize=(640, 2), meta=np.ndarray>
    tas        (time, lat, lon) float32 2GB dask.array<chunksize=(120, 320, 640), meta=np.ndarray>
Attributes: (12/36)
    Conventions:         CF-1.7 CMIP-6.2
    activity_id:         input4MIPs
    cell_measures:       area: areacella
    comment:             Based on JRA-55 reanalysis (1958-01 to 2020-07)
    contact

/scratch_local/ipykernel_2460209/2629220063.py:16: FutureWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  return xr.open_dataset(


In [6]:
# =========================
# 2. Standardize coordinate names
# =========================

COORD_ALIASES = {
    "longitude": "lon",
    "latitude": "lat",
    "xt_ocean": "lon",
    "yt_ocean": "lat",
    "nav_lon": "lon",
    "nav_lat": "lat",
}

def standardize_coords(ds):
    rename = {
        old: new
        for old, new in COORD_ALIASES.items()
        if old in ds.coords or old in ds.dims
    }
    return ds.rename(rename)

jra = standardize_coords(jra)
speedy = standardize_coords(speedy)

for name, ds in {"JRA": jra, "SPEEDY": speedy}.items():
    missing = {"time", "lat", "lon"} - set(ds.coords)
    if missing:
        raise KeyError(f"{name} is missing coordinates: {missing}")


In [7]:
# =========================
# 3. Structural comparison
# =========================

def calendar_name(time):
    return time.encoding.get(
        "calendar",
        time.attrs.get("calendar", "unknown"),
    )

def timestep_hours(time):
    if time.size < 2:
        return np.nan

    differences = np.diff(time.values)
    hours = np.array([
        d.total_seconds() / 3600.0
        if hasattr(d, "total_seconds")
        else d / np.timedelta64(1, "h")
        for d in differences
    ])
    return np.unique(hours)

def dataset_summary(ds, variable):
    da = ds[variable]
    return {
        "variable": variable,
        "dimensions": str(da.dims),
        "shape": str(da.shape),
        "dtype": str(da.dtype),
        "units": da.attrs.get("units"),
        "standard_name": da.attrs.get("standard_name"),
        "calendar": calendar_name(ds.time),
        "time_start": str(ds.time.values[0]),
        "time_end": str(ds.time.values[-1]),
        "time_records": ds.sizes["time"],
        "time_step_hours": str(timestep_hours(ds.time)),
        "lat_size": ds.sizes["lat"],
        "lat_first": float(ds.lat.values[0]),
        "lat_last": float(ds.lat.values[-1]),
        "lon_size": ds.sizes["lon"],
        "lon_first": float(ds.lon.values[0]),
        "lon_last": float(ds.lon.values[-1]),
        "fill_value": da.encoding.get("_FillValue"),
        "chunks": str(da.chunks),
    }

structural = pd.DataFrame(
    {
        "JRA": dataset_summary(jra, JRA_VARIABLE),
        "SPEEDY": dataset_summary(speedy, SPEEDY_VARIABLE),
    }
)

structural

,JRA,SPEEDY
variable,tas,tas
dimensions,"('time', 'lat', 'lon')","('time', 'lat', 'lon')"
shape,"(2920, 320, 640)","(1460, 48, 96)"
dtype,float32,float32
units,K,K
standard_name,air_temperature,air_temperature
calendar,gregorian,365_day
time_start,1989-01-01 00:00:00,1989-01-01 00:00:00
time_end,1989-12-31 21:00:00,1989-12-31 18:00:00
time_records,2920,1460


In [8]:
# =========================
# 4. Attribute comparison
# =========================

def compare_dicts(left, right):
    keys = sorted(set(left) | set(right))
    rows = []

    for key in keys:
        lv = left.get(key, "<missing>")
        rv = right.get(key, "<missing>")
        rows.append({
            "attribute": key,
            "JRA": repr(lv),
            "SPEEDY": repr(rv),
            "equal": lv == rv,
        })

    return pd.DataFrame(rows)

variable_attributes = compare_dicts(
    jra[JRA_VARIABLE].attrs,
    speedy[SPEEDY_VARIABLE].attrs,
)

global_attributes = compare_dicts(
    jra.attrs,
    speedy.attrs,
)

print("Variable attributes")
display(variable_attributes)

print("Global attributes")
display(global_attributes)

Variable attributes


,attribute,JRA,SPEEDY,equal
0,cell_measures,'area: areacella','<missing>',False
1,cell_methods,'area: mean time: point','<missing>',False
2,comment,"'near-surface (usually, 2 meter) air temperature'",'<missing>',False
3,history,"""2020-09-15T18:49:55Z altered by CMOR: Treated...",'<missing>',False
4,long_name,'Near-Surface Air Temperature','Near-surface air temperature',False
5,mapping_note,'<missing>','Provisional mapping SPEEDY ST to ACCESS-OM2 tas',False
6,source_model,'<missing>','SPEEDY',False
7,source_variable,'<missing>','ST',False
8,standard_name,'air_temperature','air_temperature',True
9,units,'K','K',True


Global attributes


,attribute,JRA,SPEEDY,equal
0,Conventions,'CF-1.7 CMIP-6.2','<missing>',False
1,activity_id,'input4MIPs','<missing>',False
2,cell_measures,'area: areacella','<missing>',False
3,cmor_version,'3.6.0','<missing>',False
4,comment,'Based on JRA-55 reanalysis (1958-01 to 2020-07)',"'Provisional NetCDF export. Metadata, coordina...",False
5,contact,'Hiroyuki Tsujino (htsujino@mri-jma.go.jp)','<missing>',False
6,creation_date,'2020-09-15T18:50:02Z','<missing>',False
7,data_specs_version,'01.00.32','<missing>',False
8,dataset_category,'atmosphericState','<missing>',False
9,external_variables,'areacella','<missing>',False


In [9]:
# =========================
# 5. Coordinate comparison
# =========================

coordinate_rows = []

for coord in ("time", "lat", "lon"):
    coordinate_rows.append({
        "coordinate": coord,
        "JRA size": jra.sizes[coord],
        "SPEEDY size": speedy.sizes[coord],
        "same size": jra.sizes[coord] == speedy.sizes[coord],
        "same values": (
            jra.sizes[coord] == speedy.sizes[coord]
            and np.array_equal(jra[coord].values, speedy[coord].values)
        ),
        "JRA attrs": dict(jra[coord].attrs),
        "SPEEDY attrs": dict(speedy[coord].attrs),
    })

coordinate_comparison = pd.DataFrame(coordinate_rows)
coordinate_comparison


,coordinate,JRA size,SPEEDY size,same size,same values,JRA attrs,SPEEDY attrs
0,time,2920,1460,False,False,"{'bounds': 'time_bnds', 'axis': 'T', 'long_nam...","{'standard_name': 'time', 'long_name': 'time',..."
1,lat,320,48,False,False,"{'bounds': 'lat_bnds', 'units': 'degrees_north...","{'standard_name': 'latitude', 'long_name': 'la..."
2,lon,640,96,False,False,"{'bounds': 'lon_bnds', 'units': 'degrees_east'...","{'standard_name': 'longitude', 'long_name': 'l..."


In [10]:
def field_statistics_tiny(da):
    sample = da.isel(
        time=slice(0, min(16, da.sizes["time"])),
        lat=slice(None, None, 16),
        lon=slice(None, None, 16,
    )

    return {
        "mean": float(sample.mean(skipna=True).compute().values),
        "min": float(sample.min(skipna=True).compute().values),
        "max": float(sample.max(skipna=True).compute().values),
    }


statistics = pd.DataFrame({
    "JRA": field_statistics_tiny(jra[JRA_VARIABLE]),
    "SPEEDY": field_statistics_tiny(speedy[SPEEDY_VARIABLE]),
})

statistics

SyntaxError: '(' was never closed (3314760430.py, line 2)

In [11]:
# =========================
# 7. Harmonize longitude and latitude
# =========================

def normalize_grid(da):
    # Normalize longitude to [0, 360).
    lon = da.lon % 360.0
    da = da.assign_coords(lon=lon).sortby("lon")

    # Remove a duplicated cyclic endpoint if present.
    _, unique_index = np.unique(
        np.round(da.lon.values, 10),
        return_index=True,
    )
    da = da.isel(lon=np.sort(unique_index))

    # Interpolation requires monotonic latitude.
    da = da.sortby("lat")
    return da

jra_tas = normalize_grid(jra[JRA_VARIABLE])
speedy_tas = normalize_grid(speedy[SPEEDY_VARIABLE])

print("JRA grid:", jra_tas.sizes["lat"], "x", jra_tas.sizes["lon"])
print("SPEEDY grid:", speedy_tas.sizes["lat"], "x", speedy_tas.sizes["lon"])


JRA grid: 320 x 640
SPEEDY grid: 48 x 96


In [12]:
# =========================
# 8. Put the fields on one grid
# =========================

if REGRID_TO == "jra":
    reference = jra_tas
    candidate = speedy_tas.interp(
        lat=jra_tas.lat,
        lon=jra_tas.lon,
        method="linear",
    )
    reference_name = "JRA"
    candidate_name = "SPEEDY interpolated to JRA"
elif REGRID_TO == "speedy":
    reference = speedy_tas
    candidate = jra_tas.interp(
        lat=speedy_tas.lat,
        lon=speedy_tas.lon,
        method="linear",
    )
    reference_name = "SPEEDY"
    candidate_name = "JRA interpolated to SPEEDY"
else:
    raise ValueError("REGRID_TO must be 'jra' or 'speedy'")

print(reference_name, reference.shape)
print(candidate_name, candidate.shape)


JRA (2920, 320, 640)
SPEEDY interpolated to JRA (1460, 320, 640)


In [13]:
# =========================
# 9. Match the RYF cycle by cycle position
# =========================

def cycle_key(time):
    return (
        time.dt.month.astype(str).str.zfill(2)
        + "-"
        + time.dt.day.astype(str).str.zfill(2)
        + "T"
        + time.dt.hour.astype(str).str.zfill(2)
    )

reference = reference.assign_coords(
    cycle_key=("time", cycle_key(reference.time).data)
)
candidate = candidate.assign_coords(
    cycle_key=("time", cycle_key(candidate.time).data)
)

# Average duplicates if one input contains more than one year.
reference_cycle = reference.groupby("cycle_key").mean("time")
candidate_cycle = candidate.groupby("cycle_key").mean("time")

reference_cycle, candidate_cycle = xr.align(
    reference_cycle,
    candidate_cycle,
    join="inner",
)

print("Matched cycle records:", reference_cycle.sizes["cycle_key"])


Matched cycle records: 1460


In [ ]:
# =========================
# 10. Scientific difference metrics
# =========================

difference = candidate_cycle - reference_cycle

metrics = xr.Dataset({
    "bias": difference.mean(),
    "mae": abs(difference).mean(),
    "rmse": np.sqrt((difference ** 2).mean()),
    "pattern_correlation": xr.corr(
        candidate_cycle.stack(points=("cycle_key", "lat", "lon")),
        reference_cycle.stack(points=("cycle_key", "lat", "lon")),
        dim="points",
    ),
}).compute()

metrics


In [ ]:
# =========================
# 11. Mean spatial fields
# =========================

reference_mean = reference_cycle.mean("cycle_key").compute()
candidate_mean = candidate_cycle.mean("cycle_key").compute()
difference_mean = difference.mean("cycle_key").compute()

for field, title in [
    (reference_mean, f"{reference_name}: annual mean tas"),
    (candidate_mean, f"{candidate_name}: annual mean tas"),
    (difference_mean, "SPEEDY minus JRA: annual mean tas difference"),
]:
    plt.figure(figsize=(11, 4.5))
    field.plot()
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# 12. Global-mean annual cycle
# =========================

def area_weighted_mean(da):
    weights = np.cos(np.deg2rad(da.lat))
    return da.weighted(weights).mean(("lat", "lon"))

reference_series = area_weighted_mean(reference_cycle).compute()
candidate_series = area_weighted_mean(candidate_cycle).compute()

plt.figure(figsize=(12, 4.5))
plt.plot(reference_series.values, label=reference_name)
plt.plot(candidate_series.values, label=candidate_name)
plt.xlabel("6-hourly position in repeat-year cycle")
plt.ylabel("tas [K]")
plt.title("Area-weighted global-mean repeat-year cycle")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# 13. Save comparison tables
# =========================

REPORT_DIR = Path.cwd() / "tas_comparison_report"
REPORT_DIR.mkdir(exist_ok=True)

structural.to_csv(REPORT_DIR / "structural_comparison.csv")
variable_attributes.to_csv(
    REPORT_DIR / "variable_attributes.csv",
    index=False,
)
global_attributes.to_csv(
    REPORT_DIR / "global_attributes.csv",
    index=False,
)
coordinate_comparison.to_csv(
    REPORT_DIR / "coordinate_comparison.csv",
    index=False,
)
statistics.to_csv(REPORT_DIR / "basic_statistics.csv")

with (REPORT_DIR / "difference_metrics.json").open("w") as f:
    json.dump(
        {
            name: float(metrics[name].values)
            for name in metrics.data_vars
        },
        f,
        indent=2,
    )

print("Saved report to:", REPORT_DIR.resolve())